# 15. Общая оценка всех моделей (Kaggle)

Собирает прогнозы LightGBM, графовых сетей и Chronos-2, пересчитывает бейзлайны (с HA) и оценивает всё
по протоколу на val и test. Дополнительно — срезы, близкие к литературе, точность по уровням агрегации,
разброс по коротким окнам и значимость разниц между парами моделей.

**Input:** датасет `vkr-transit-data` и вывод ноутбуков `lgbm-final`, `gnn`, `chronos2` (Your Work → Notebooks).
Accelerator — None, Internet — On. Время ~1 ч. Результат — `results_csv.zip` в Output.

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## 1. Прогнозы всех моделей и бейзлайны

In [ ]:
import pandas as pd
for p in sorted(glob.glob("/kaggle/input/**/results/preds/*.npz", recursive=True)):
    dst = f"{R}/preds/{os.path.basename(p)}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
logs = glob.glob("/kaggle/input/**/results/runs.csv", recursive=True)
if logs:
    pd.concat([pd.read_csv(p) for p in logs]).drop_duplicates().to_csv(f"{R}/runs.csv", index=False)
for p in glob.glob("/kaggle/input/**/results/fi_*.csv", recursive=True):
    shutil.copy(p, R)
!cd {REPO} && python src/baselines.py
for s in ["val", "test"]:
    print(s, sorted(os.path.basename(p).split("__")[0] for p in glob.glob(f"{R}/preds/*__{s}.npz")))

## 2. Оценка по протоколу

In [ ]:
!cd {REPO} && python src/evaluate.py --split val --ref norm_level

In [ ]:
!cd {REPO} && python src/evaluate.py --split test --ref norm_level

## 3. Срезы, агрегация, короткие окна, пары моделей

In [ ]:
PAIRS = " ".join([
    "lgbm_all:lgbm", "lgbm_wactual:lgbm", "lgbm_final:lgbm", "lgbm_bc:lgbm",      # соседи, погода, поправка
    "gnn_od_in:gnn_none", "gnn_all:gnn_none", "gnn_od_dyn:gnn_none", "gnn_adapt:gnn_none",
    "gnn_line:gnn_none", "gnn_dist:gnn_none",                                       # графы в сети (абляция)
    "lgbm:gnn_od_in", "lgbm:chronos2", "chronos2_cov:chronos2", "lgbm:ha",
])
!cd {REPO} && python src/report.py --split val --pairs {PAIRS}

In [ ]:
!cd {REPO} && python src/report.py --split test --pairs {PAIRS}

## 4. Архив

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/*.csv"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*.csv")))